## Imports

In [1]:
import wandb
import logging
from tqdm import tqdm
from wandb.sdk.wandb_run import Run
import numpy as np
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objs as go
import seaborn as sns
import matplotlib.pyplot as plt
from nn_core.common import PROJECT_ROOT
import json

/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


## Configuration

In [2]:
from mass.utils.plots import Palette

plt.rcParams.update(
    {
        "text.usetex": True,
        "font.family": "serif",
        "axes.titlesize": 24,        # Larger axes/title fonts
        "axes.labelsize": 24,
        "xtick.labelsize": 24,
        "ytick.labelsize": 20,
        "legend.fontsize": 24,
    }
)
sns.set_context("talk")

cmap_name = "coolwarm_r"

palette = Palette(f"{PROJECT_ROOT}/misc/palette.json", map_path=f"{PROJECT_ROOT}/misc/palette_map.json")
palette

Project not installed in the current env, activate the correct env or install it with:
	pip install -e .


{'blue': '#335c67',
 'white': '#fff3b0',
 'yellow': '#e09f3e',
 'red': '#9e2a2b',
 'dark red': '#540b0e',
 'green': '#81b29a'}

## Get runs

In [3]:
api = wandb.Api()
entity, project = "gladia", "mass"  # set to your entity and project

In [4]:
def get_runs(entity, project, positive_tags, negative_tags):
    filters_pos_tags = {"$and": [{"tags": {"$eq": pos_tag}} for pos_tag in positive_tags]}
    filters_neg_tags = {}

    print(filters_pos_tags)
    filters = {**filters_pos_tags, **filters_neg_tags}
    runs = api.runs(entity + "/" + project, filters=filters)

    print(f"There are {len(runs)} runs respecting these conditions.")
    return runs

In [5]:
tags = [
    "ZeroShot"
]  

In [6]:
runs = get_runs(entity, project, positive_tags=tags, negative_tags=[])

{'$and': [{'tags': {'$eq': 'ZeroShot'}}]}
There are 9 runs respecting these conditions.


In [7]:
models = ['ViT-B-32', 'ViT-B-16', 'ViT-L-14']

In [8]:
ref_run = runs[0]

In [9]:
print(set(ref_run.history().columns))

{'normalized_acc/test/GTSRB', 'normalized_acc/test/EuroSAT', 'acc/test/SUN397', 'loss/test/SUN397', 'normalized_acc/test/SUN397', 'normalized_acc/test/CIFAR100', 'acc/test/PCAM', 'loss/test/Flowers102', '_runtime', 'normalized_acc/test/avg', 'loss/test/DTD', 'loss/test/FER2013', 'acc/test/SVHN', 'loss/test/GTSRB', 'normalized_acc/test/MNIST', 'normalized_acc/test/Cars', 'acc/test/EuroSAT', 'acc/test/FER2013', 'epoch', 'acc/test/DTD', 'radar', 'normalized_acc/test/SVHN', 'loss/test/EuroSAT', 'loss/test/MNIST', 'normalized_acc/test/FER2013', 'acc/test/CIFAR100', 'loss/test/STL10', 'acc/test/Cars', '_step', 'loss/test/OxfordIIITPet', 'normalized_acc/test/RESISC45', 'loss/test/Cars', 'acc/test/OxfordIIITPet', 'normalized_acc/test/Flowers102', 'normalized_acc/test/OxfordIIITPet', 'loss/test/SVHN', 'acc/test/MNIST', '_timestamp', 'normalized_acc/test/STL10', 'acc/test/Flowers102', 'normalized_acc/test/PCAM', 'acc/test/RESISC45', 'trainer/global_step', 'loss/test/RESISC45', 'acc/test/GTSRB', 

In [10]:
print(ref_run.config['core/tags'])

['static_merge', 'n14', 'ViT-B-32']


#### Hparams

In [11]:
benchmarks = ['n8', 'n14', 'n20']
models = ['ViT-B-32', 'ViT-B-16', 'ViT-L-14']

In [12]:
avg_accs = {model: {benchmark: {'avg_acc': 0.0, 'norm_acc': 0.0} for benchmark in benchmarks} for model in models}

for run in runs:
    model = run.config['nn/encoder/model_name']

    try:
        N = run.config['num_tasks']
    except KeyError:
        N = run.config['ntasks']

    benchmark = f'n{N}'

    avg_accs[model][benchmark]['avg_acc'] = run.summary['acc/test/avg']
    avg_accs[model][benchmark]['norm_acc'] = run.summary['normalized_acc/test/avg']

In [13]:
avg_accs

{'ViT-B-32': {'n8': {'avg_acc': 0.48119574412703514,
   'norm_acc': 0.5482873134315014},
  'n14': {'avg_acc': 0.5694975405931473, 'norm_acc': 0.6452069516692843},
  'n20': {'avg_acc': 0.5752051994204521, 'norm_acc': 0.6521049853414297}},
 'ViT-B-16': {'n8': {'avg_acc': 0.5531878359615803,
   'norm_acc': 0.6060628853738308},
  'n14': {'avg_acc': 0.6193169142518725, 'norm_acc': 0.6792114696332386},
  'n20': {'avg_acc': 0.6250154849141836, 'norm_acc': 0.6831165432929993}},
 'ViT-L-14': {'n8': {'avg_acc': 0.6491173133254051,
   'norm_acc': 0.6921657919883728},
  'n14': {'avg_acc': 0.6908513626882008, 'norm_acc': 0.7380124798842839},
  'n20': {'avg_acc': 0.6824994001537561, 'norm_acc': 0.7265810780227184}}}

In [14]:
# print latex

row = '& '
for model in models:

    for benchmark in benchmarks:
        avg_acc = avg_accs[model][benchmark]['avg_acc']
        norm_acc = avg_accs[model][benchmark]['norm_acc']

        row += f"${avg_acc*100:.1f}_{{({norm_acc*100:.1f})}}$ & "


print(row[:-2] + '\\\\')


& $48.1_{(54.8)}$ & $56.9_{(64.5)}$ & $57.5_{(65.2)}$ & $55.3_{(60.6)}$ & $61.9_{(67.9)}$ & $62.5_{(68.3)}$ & $64.9_{(69.2)}$ & $69.1_{(73.8)}$ & $68.2_{(72.7)}$ \\
